# E13 + E18 — Differentiable simulation vs PPO in the clean physics (GPU runs)

Redoes the E11 comparison in the physics fixed by E12–E15 (`sub_step = 20`, no velocity
clip, `water_drag = 0.005`) with the final reward of E16 (smoothness penalty λ = 1),
at equal simulation budget (`NB` episodes of train2 = `NB` PPO iterations).

* **Part A (E13, pilot):** which train2 variant? historical truncation vs full
  truncation vs per-window updates, seed 0, 100 episodes each.
* **Part B (E18):** the chosen train2 variant vs PPO, 3 seeds each, `NB = 400`.

Every output goes to **Google Drive**. Each training cell refuses to start if Drive is
not mounted. If the runtime disconnects: rerun cells 1–3, then continue from the first
run whose checkpoint is missing on Drive.

**Before starting:** `Runtime → Change runtime type → GPU`.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive (persistent outputs)

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/watercrea/E18"
!mkdir -p {OUT}

def verifier_drive():
    assert os.path.ismount("/content/drive"), "Drive pas monté : relance la cellule 2"
    print("Drive OK ->", OUT)

## 3. Code

In [ ]:
%cd /content
!rm -rf watercrea && git clone -q https://github.com/mrlrns/parallel-neuroevolution-engine.git watercrea
%cd /content/watercrea
!git log --oneline -1
!pip install -q pyyaml matplotlib
CHAMP = "champion_raffine/reference.pt"
COMMUN = "--poids-aleatoires --coef-regularite 1 --physique-compilee"

## 4. Environment check (must print ✅)

CPU on purpose (exact equality test, see E11).

In [ ]:
!python -m baselines.check_env --chemin-champion {CHAMP} --train2-batch-size 200

## 5. Timing test

3 episodes of each method into a scratch folder. One full run ≈ time × NB / 3.
Compilation makes the first episode slower: the estimate is pessimistic.

**Out of memory?** With 1 update per episode (`hist`, `complet`) the graph of the whole
episode (300 frames × 20 sub-steps) is kept until `backward`. If train2 crashes with
`CUDA out of memory`, add `--train2-batch-size 1000 --ppo-n-envs 1000` to `COMMUN`
(same value for both methods, so the budget stays equal) and rerun this cell.

In [ ]:
!time python train2.py --chemin-champion {CHAMP} {COMMUN} --nb-episodes 3 --dossier-runs /tmp/t --dossier-champion-raffine /tmp/t
!time python train2.py --chemin-champion {CHAMP} {COMMUN} --maj-par-fenetre --nb-episodes 3 --dossier-runs /tmp/t --dossier-champion-raffine /tmp/t
!time python -m baselines.ppo --chemin-champion {CHAMP} {COMMUN} --ppo-nb-iterations 3 --dossier-runs /tmp/t --dossier-baselines /tmp/t

## Part A — E13 pilot: which train2 variant? (seed 0, 100 episodes each)

One variable at a time:
* `hist`: historical truncation (detaches X, Y, vX, vY only), 1 update / episode
* `complet`: full truncation (`--detachement-complet`), 1 update / episode
* `fenetre`: per-window updates (`--maj-par-fenetre`, implies full truncation), 30 updates / episode

In [ ]:
verifier_drive()
!python train2.py --chemin-champion {CHAMP} {COMMUN} --seed 0 --nb-episodes 100 --dossier-runs {OUT}/A_hist/runs --dossier-champion-raffine {OUT}/A_hist/diffsim

In [ ]:
verifier_drive()
!python train2.py --chemin-champion {CHAMP} {COMMUN} --detachement-complet --seed 0 --nb-episodes 100 --dossier-runs {OUT}/A_complet/runs --dossier-champion-raffine {OUT}/A_complet/diffsim

In [ ]:
verifier_drive()
!python train2.py --chemin-champion {CHAMP} {COMMUN} --maj-par-fenetre --seed 0 --nb-episodes 100 --dossier-runs {OUT}/A_fenetre/runs --dossier-champion-raffine {OUT}/A_fenetre/diffsim

### Pilot verdict

Prints the last training distances of each variant. **Paste the output to Claude**
before Part B: the variant with the best final distance becomes `VARIANTE` below.

In [ ]:
import glob
import json

for v in ("hist", "complet", "fenetre"):
    for f in sorted(glob.glob(f"{OUT}/A_{v}/runs/*.jsonl")):
        recs = [json.loads(ligne) for ligne in open(f) if "_run_config" not in ligne]
        d = [r["distance_moyenne"] for r in recs if r.get("distance_moyenne") is not None]
        print(f"{v:8s} {len(d):4d} épisodes | distance début {sum(d[:5])/5:7.2f} | fin (moy. 10 derniers) {sum(d[-10:])/10:7.2f}")

## Part B — E18: chosen train2 variant vs PPO, 3 seeds, NB = 400

Set `VARIANTE` to the flags of the pilot winner: `""` (hist), `"--detachement-complet"`
or `"--maj-par-fenetre"`.

In [ ]:
NB = 400
VARIANTE = ""   # <- à remplir après le pilote

In [ ]:
verifier_drive()
!python train2.py --chemin-champion {CHAMP} {COMMUN} {VARIANTE} --seed 0 --nb-episodes {NB} --dossier-runs {OUT}/B/runs --dossier-champion-raffine {OUT}/B/diffsim

In [ ]:
verifier_drive()
!python -m baselines.ppo --chemin-champion {CHAMP} {COMMUN} --seed 0 --ppo-nb-iterations {NB} --dossier-runs {OUT}/B/runs --dossier-baselines {OUT}/B/ppo

In [ ]:
verifier_drive()
!python train2.py --chemin-champion {CHAMP} {COMMUN} {VARIANTE} --seed 1 --nb-episodes {NB} --dossier-runs {OUT}/B/runs --dossier-champion-raffine {OUT}/B/diffsim

In [ ]:
verifier_drive()
!python -m baselines.ppo --chemin-champion {CHAMP} {COMMUN} --seed 1 --ppo-nb-iterations {NB} --dossier-runs {OUT}/B/runs --dossier-baselines {OUT}/B/ppo

In [ ]:
verifier_drive()
!python train2.py --chemin-champion {CHAMP} {COMMUN} {VARIANTE} --seed 2 --nb-episodes {NB} --dossier-runs {OUT}/B/runs --dossier-champion-raffine {OUT}/B/diffsim

In [ ]:
verifier_drive()
!python -m baselines.ppo --chemin-champion {CHAMP} {COMMUN} --seed 2 --ppo-nb-iterations {NB} --dossier-runs {OUT}/B/runs --dossier-baselines {OUT}/B/ppo

## 7. Learning curves vs simulation budget

In [ ]:
from IPython.display import Image

!python -m baselines.compare --diffsim "{OUT}/B/runs/train_phase2_seed*.jsonl" --ppo "{OUT}/B/runs/ppo_seed*.jsonl" --sortie {OUT}/E18_comparison.png
Image(f"{OUT}/E18_comparison.png")

## 8. Final evaluation — same judge for every checkpoint

20 seeds × 1000 frames per checkpoint (`evaluate_seeds.py`).

In [ ]:
checkpoints = sorted(glob.glob(f"{OUT}/B/diffsim/FINAL_seed*.pt")) + sorted(glob.glob(f"{OUT}/B/ppo/ppo_seed*_best.pt"))
for ckpt in checkpoints:
    print("=" * 80, "\n", ckpt)
    !python evaluate_seeds.py --chemin-champion "{ckpt}" --nb-seeds 20 | tail -8

## 9. Flush Drive before closing

In [ ]:
drive.flush_and_unmount()